# 🐍 PythonQuest — Learn Python by Playing!

### Welcome, brave coder! 🚀

This is a **game**, not a boring lesson. You'll travel through **7 worlds** and
**27 levels**, starting from your very first `print()` and ending by building a
**real Quiz Bot game** all by yourself. Meet **Pixel** 🤖, your robot guide!

**How to play — only 2 steps:**

1. ▶️ Run **STEP 1** below once (it loads the game). Press the little **play
   button** on the left of the cell, or click it and press **Shift + Enter**.
2. 🕹️ Run **STEP 2** to open your game panel. Type code in the black box and
   press **▶ RUN & CHECK**!

> 💡 **Tip:** Stuck on a level? Tap **💡 Hint**. Really stuck? **👀 Show Answer**
> appears after a hint — but try typing it yourself, that's how your fingers
> learn! Your progress **saves automatically**, so you can close Colab and come
> back later.

---


In [ ]:
# @title 🎮 STEP 1 — Load the game  { display-mode: "form" }
# Just press the ▶ play button on the left of this cell. You don't need to
# read this code — it's the game engine. (Curious kids: peek all you like!)

# =====================================================================
#  🐍 PYTHONQUEST  —  Learn Python by Playing (Google Colab edition)
#  A level-by-level coding adventure engine for kids.
#  Everything runs inside one friendly game panel. Just press RUN.
# =====================================================================
import io, json, os, re, html, contextlib, traceback, random

try:
    import ipywidgets as W
    from IPython.display import display, HTML, clear_output
    _HAS_WIDGETS = True
except Exception:                       # pragma: no cover - non-Colab fallback
    _HAS_WIDGETS = False

# Colab needs this to show custom widgets nicely.
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

SAVE_PATH = "/content/pythonquest_save.json"
if not os.path.isdir("/content"):        # so it also works off-Colab
    SAVE_PATH = os.path.join(os.path.expanduser("~"), ".pythonquest_save.json")


# ---------------------------------------------------------------------
#  Small helpers used by the level checkers
# ---------------------------------------------------------------------
def norm(text):
    """Lowercase, trim, squeeze spaces — for forgiving text comparison."""
    return re.sub(r"\s+", " ", str(text).strip().lower())

def lines(text):
    return [ln.rstrip() for ln in str(text).splitlines() if ln.strip() != ""]

def ok(msg="Perfect!"):
    return True, msg

def no(msg):
    return False, msg


# ---------------------------------------------------------------------
#  Friendly translations of scary Python errors
# ---------------------------------------------------------------------
def friendly_error(exc):
    name = type(exc).__name__
    detail = html.escape(str(exc))
    tips = {
        "SyntaxError":     "Python couldn't read your code. Check for a missing "
                           ": at the end of a line, or an unmatched ( ) or \" \".",
        "IndentationError":"Your spacing is off. Lines inside an if/for/def need to "
                           "be pushed in (indented) by 4 spaces.",
        "NameError":       "You used a word Python doesn't know yet. Did you spell a "
                           "variable name the same way you made it? Are your quotes missing?",
        "TypeError":       "You mixed things that don't fit — like adding a number to a "
                           "word. Try str() or int() to convert first.",
        "ValueError":      "The value doesn't fit what was expected — e.g. turning the "
                           "word \"cat\" into a number with int().",
        "ZeroDivisionError":"You divided by zero. Nothing can be split into 0 pieces!",
        "IndexError":      "You asked for an item that isn't there. Remember lists start "
                           "counting at 0.",
        "KeyError":        "That key isn't in the dictionary. Check the spelling of the key.",
        "AttributeError":  "That thing can't do that action. Check the method name.",
    }
    tip = tips.get(name, "Read the message carefully — it usually points at the line.")
    return name, detail, tip


# =====================================================================
#  THE GAME ENGINE
# =====================================================================
class PythonQuest:
    RANKS = [
        (0,   "🐣 Hatchling Coder"),
        (60,  "🐍 Baby Python"),
        (150, "🦎 Code Explorer"),
        (280, "🐲 Loop Wizard"),
        (450, "🦅 Function Master"),
        (650, "🚀 Python Hero"),
        (900, "👑 Grand Code Champion"),
    ]

    def __init__(self, levels, player=None):
        self.levels = levels
        self.state = self._load()
        if player:
            self.state["player"] = player
        # persistent code workspace shared across levels (feels like one program)
        self.ws = {}
        self.hint_i = 0
        self.revealed = False
        self.screen = W.Output() if _HAS_WIDGETS else None

    # ---------- save / load ----------
    def _fresh(self):
        return {"player": "Coder", "xp": 0, "level": 0,
                "done": [], "badges": []}

    def _load(self):
        try:
            with open(SAVE_PATH) as f:
                s = json.load(f)
                for k, v in self._fresh().items():
                    s.setdefault(k, v)
                return s
        except Exception:
            return self._fresh()

    def _save(self):
        try:
            with open(SAVE_PATH, "w") as f:
                json.dump(self.state, f)
        except Exception:
            pass

    def reset(self):
        """Wipe all progress and start the adventure over."""
        self.state = self._fresh()
        self.ws = {}
        self._save()
        print("🧹 Progress reset! Run  quest.start()  to begin again.")

    # ---------- rank / progress ----------
    def _rank(self):
        r = self.RANKS[0][1]
        for need, title in self.RANKS:
            if self.state["xp"] >= need:
                r = title
        return r

    def _next_rank_need(self):
        for need, _ in self.RANKS:
            if self.state["xp"] < need:
                return need
        return self.RANKS[-1][0]

    # =================================================================
    #  PUBLIC ENTRY POINTS
    # =================================================================
    def start(self):
        if not _HAS_WIDGETS:
            print("This game needs ipywidgets (built into Google Colab).")
            return
        self._inject_css()
        display(self.screen)
        self._render()

    def map(self):
        """Show the whole adventure map and what you've unlocked."""
        if not _HAS_WIDGETS:
            return
        self._inject_css()
        rows, world = [], None
        for i, lv in enumerate(self.levels):
            if lv["world"] != world:
                world = lv["world"]
                rows.append(f"<div class='pq-world'>{html.escape(world)}</div>")
            if i in self.state["done"]:
                mark, cls = "✅", "pq-cell done"
            elif i == self.state["level"]:
                mark, cls = "▶️", "pq-cell now"
            elif i <= self.state["level"]:
                mark, cls = "🔓", "pq-cell open"
            else:
                mark, cls = "🔒", "pq-cell lock"
            rows.append(
                f"<div class='{cls}'>{mark} <b>{i+1}.</b> "
                f"{html.escape(lv['title'])}</div>")
        badges = " ".join(self.state["badges"]) or "—"
        display(HTML(
            f"<div class='pq-card pq-map'><h2>🗺️ Your Quest Map</h2>"
            f"<div class='pq-sub'>Rank: <b>{self._rank()}</b> &nbsp;•&nbsp; "
            f"XP: <b>{self.state['xp']}</b> &nbsp;•&nbsp; Badges: {badges}</div>"
            + "".join(rows) + "</div>"))

    # =================================================================
    #  RENDERING A LEVEL
    # =================================================================
    def _render(self):
        self.hint_i = 0
        self.revealed = False
        with self.screen:
            clear_output(wait=True)

            if self.state["level"] >= len(self.levels):
                display(HTML(self._certificate()))
                return

            i = self.state["level"]
            lv = self.levels[i]

            display(HTML(self._header(i, lv)))
            display(HTML(self._story(lv)))
            display(HTML(self._concept(lv)))

            code = W.Textarea(
                value=lv.get("starter", "# write your code here\n"),
                layout=W.Layout(width="100%", height="150px"))
            code.add_class("pq-code")
            self._code = code

            run   = W.Button(description="▶  RUN & CHECK", button_style="success")
            hint  = W.Button(description="💡 Hint")
            ans   = W.Button(description="👀 Show Answer")
            skip  = W.Button(description="⏭ Skip")
            back  = W.Button(description="🗺 Map")
            run.add_class("pq-btn"); hint.add_class("pq-btn")
            ans.add_class("pq-btn"); skip.add_class("pq-btn"); back.add_class("pq-btn")

            run.on_click(self._on_run)
            hint.on_click(self._on_hint)
            ans.on_click(self._on_answer)
            skip.on_click(self._on_skip)
            back.on_click(lambda b: self.map())

            self._out = W.Output()
            display(W.HTML("<div class='pq-label'>✏️ Your code:</div>"))
            display(code)
            display(W.HBox([run, hint, ans, skip, back]))
            display(self._out)

    # ---------- HTML blocks ----------
    def _header(self, i, lv):
        pct = int(len(self.state["done"]) / len(self.levels) * 100)
        nxt = self._next_rank_need()
        return (
            f"<div class='pq-card pq-head'>"
            f"<div class='pq-headtop'>"
            f"<div><span class='pq-lvl'>LEVEL {i+1} / {len(self.levels)}</span>"
            f"<h1>{html.escape(lv['title'])}</h1>"
            f"<div class='pq-world2'>{html.escape(lv['world'])}</div></div>"
            f"<div class='pq-avatar'>{lv.get('emoji','🐍')}</div></div>"
            f"<div class='pq-sub'>👤 {html.escape(self.state['player'])} "
            f"&nbsp;•&nbsp; {self._rank()} &nbsp;•&nbsp; "
            f"⭐ {self.state['xp']} XP</div>"
            f"<div class='pq-bar'><div class='pq-fill' style='width:{pct}%'>"
            f"{pct}%</div></div></div>")

    def _story(self, lv):
        return (f"<div class='pq-card pq-story'>"
                f"<b>🤖 Pixel says:</b> {lv['story']}</div>")

    def _concept(self, lv):
        return (f"<div class='pq-card pq-concept'>"
                f"<h3>📚 What to learn</h3>{lv['concept']}"
                f"<div class='pq-task'><b>🎯 Your mission:</b><br>{lv['task']}</div>"
                f"</div>")

    # ---------- button handlers ----------
    def _on_run(self, b):
        lv = self.levels[self.state["level"]]
        src = self._code.value
        with self._out:
            clear_output(wait=True)
            buf = io.StringIO()
            self.ws["input"] = _mock_input(lv.get("inputs", []))
            try:
                with contextlib.redirect_stdout(buf), _loop_guard():
                    exec(src, self.ws)
            except _LoopRunaway:                 # infinite / runaway loop
                display(HTML(
                    "<div class='pq-card pq-err'>"
                    "<b>🔁 Whoa! Your loop ran WAY too many times.</b><br>"
                    "I stopped it so nothing freezes. This usually means a "
                    "<code>while</code> loop never stops — did you forget to "
                    "change the value that ends it? "
                    "(for example <code>n = n - 1</code> inside the loop)</div>"))
                return
            except Exception as e:               # kid-friendly error
                name, detail, tip = friendly_error(e)
                display(HTML(
                    f"<div class='pq-card pq-err'>"
                    f"<b>😅 Oops — {name}</b><br>"
                    f"<code>{detail}</code><br><br>💡 {tip}</div>"))
                return

            out = buf.getvalue()
            if out.strip():
                display(HTML("<div class='pq-out'><b>🖥️ Output:</b><pre>"
                             + html.escape(out) + "</pre></div>"))

            try:
                passed, msg = lv["check"](self.ws, out)
            except Exception:
                passed, msg = False, ("Almost! Something in your answer wasn't "
                                      "quite what the mission asked for.")

            if passed:
                self._win(lv, msg)
            else:
                display(HTML(f"<div class='pq-card pq-try'>"
                             f"<b>🤔 Not yet!</b> {msg}<br>"
                             f"<small>Tweak your code and press RUN again. "
                             f"Stuck? Tap 💡 Hint.</small></div>"))

    def _win(self, lv, msg):
        i = self.state["level"]
        first = i not in self.state["done"]
        gained = lv.get("xp", 20) if first else 0
        if first:
            self.state["done"].append(i)
            self.state["xp"] += gained
            if lv.get("badge") and lv["badge"] not in self.state["badges"]:
                self.state["badges"].append(lv["badge"])
            self._save()

        conf = "".join(random.choice("🎉✨🎊⭐🌟💫") for _ in range(10))
        display(HTML(
            f"<div class='pq-card pq-win'><div class='pq-conf'>{conf}</div>"
            f"<h2>✅ Level Complete!</h2><p>{msg}</p>"
            f"<p>{'⭐ +' + str(gained) + ' XP' if gained else 'Replayed — no new XP'} "
            f"&nbsp;•&nbsp; Rank: <b>{self._rank()}</b>"
            + (f"<br>🏅 New badge: <b>{lv['badge']}</b>" if first and lv.get('badge') else "")
            + "</p></div>"))

        nxt = W.Button(description="➡  NEXT LEVEL", button_style="primary")
        nxt.add_class("pq-btn")
        nxt.on_click(self._on_next)
        display(nxt)

    def _on_next(self, b):
        if self.state["level"] < len(self.levels):
            self.state["level"] += 1
            self._save()
        self._render()

    def _on_skip(self, b):
        self._on_next(b)

    def _on_hint(self, b):
        lv = self.levels[self.state["level"]]
        hints = lv.get("hints", [])
        with self._out:
            clear_output(wait=True)
            if not hints:
                display(HTML("<div class='pq-card pq-hint'>No hints for this one — "
                             "you've got it! 💪</div>"))
                return
            h = hints[min(self.hint_i, len(hints) - 1)]
            display(HTML(f"<div class='pq-card pq-hint'>💡 <b>Hint "
                         f"{min(self.hint_i+1, len(hints))}:</b> {h}</div>"))
            self.hint_i = min(self.hint_i + 1, len(hints))

    def _on_answer(self, b):
        lv = self.levels[self.state["level"]]
        with self._out:
            clear_output(wait=True)
            if not self.revealed and self.hint_i < 1:
                display(HTML("<div class='pq-card pq-hint'>Try a 💡 Hint first! "
                             "The answer will unlock after a hint.</div>"))
                self.hint_i = 1
                return
            self.revealed = True
            display(HTML(
                "<div class='pq-card pq-ans'><b>👀 One good answer:</b>"
                f"<pre>{html.escape(lv.get('answer','(no answer provided)'))}</pre>"
                "<small>Type it into the code box yourself, then press RUN — "
                "your fingers remember what your eyes forget!</small></div>"))

    # ---------- finale ----------
    def _certificate(self):
        badges = " ".join(self.state["badges"]) or "🏅"
        return (
            f"<div class='pq-card pq-cert'>"
            f"<div class='pq-conf'>🎉✨🎊🌟💫⭐🎉✨🎊🌟</div>"
            f"<h1>🏆 QUEST COMPLETE! 🏆</h1>"
            f"<h2>{html.escape(self.state['player'])}</h2>"
            f"<p>has journeyed through all {len(self.levels)} levels of PythonQuest</p>"
            f"<p class='pq-big'>{self._rank()}</p>"
            f"<p>⭐ {self.state['xp']} XP &nbsp;•&nbsp; Badges: {badges}</p>"
            f"<hr><p>You can now read code, use variables, make decisions, "
            f"loop, store data in lists &amp; dictionaries, and build your own "
            f"functions and programs. Go build something real! 🚀</p>"
            f"<small>Run <code>quest.reset()</code> to play again, or "
            f"<code>quest.map()</code> to see your map.</small></div>")

    # ---------- styling ----------
    def _inject_css(self):
        display(HTML("""
<style>
.pq-card{font-family:'Segoe UI',system-ui,sans-serif;border-radius:18px;
  padding:18px 22px;margin:12px 0;box-shadow:0 6px 18px rgba(0,0,0,.12);
  line-height:1.5;color:#1f2937;}
.pq-head{background:linear-gradient(135deg,#7c3aed,#db2777);color:#fff;}
.pq-head h1{margin:2px 0;font-size:26px;}
.pq-headtop{display:flex;justify-content:space-between;align-items:center;}
.pq-lvl{background:rgba(255,255,255,.25);padding:3px 10px;border-radius:20px;
  font-size:12px;font-weight:700;letter-spacing:1px;}
.pq-world2{opacity:.9;font-weight:600;}
.pq-avatar{font-size:52px;}
.pq-sub{margin-top:8px;font-size:14px;opacity:.95;}
.pq-bar{background:rgba(255,255,255,.3);border-radius:20px;height:20px;
  margin-top:10px;overflow:hidden;}
.pq-fill{background:#fde047;color:#78350f;height:100%;text-align:center;
  font-size:12px;font-weight:700;border-radius:20px;transition:width .4s;}
.pq-story{background:#ecfeff;border-left:6px solid #06b6d4;}
.pq-concept{background:#fffbeb;border-left:6px solid #f59e0b;}
.pq-concept h3{margin:0 0 8px;color:#b45309;}
.pq-concept code,.pq-story code{background:#1f2937;color:#a7f3d0;padding:1px 6px;
  border-radius:6px;font-size:13px;}
.pq-concept pre{background:#1f2937;color:#e5e7eb;padding:12px;border-radius:10px;
  overflow:auto;font-size:13px;}
.pq-task{background:#fef3c7;border-radius:10px;padding:10px 12px;margin-top:10px;}
.pq-label{font-weight:700;margin:6px 0;font-family:'Segoe UI',sans-serif;color:#374151;}
.pq-code textarea{font-family:'Fira Mono','Consolas',monospace!important;
  font-size:14px!important;background:#0f172a!important;color:#e2e8f0!important;
  border-radius:12px!important;padding:12px!important;line-height:1.5!important;}
.pq-btn{margin:6px 6px 6px 0!important;border-radius:10px!important;
  font-weight:700!important;}
.pq-out{background:#0f172a;color:#a7f3d0;border-radius:12px;padding:10px 14px;
  margin:8px 0;font-family:'Segoe UI',sans-serif;}
.pq-out pre{margin:6px 0 0;color:#e2e8f0;white-space:pre-wrap;}
.pq-win{background:linear-gradient(135deg,#22c55e,#16a34a);color:#fff;text-align:center;}
.pq-win h2{margin:6px 0;}
.pq-conf{font-size:24px;letter-spacing:4px;}
.pq-try{background:#fef2f2;border-left:6px solid #ef4444;}
.pq-err{background:#fef2f2;border-left:6px solid #dc2626;}
.pq-err code{background:#1f2937;color:#fca5a5;padding:2px 6px;border-radius:6px;}
.pq-hint{background:#eff6ff;border-left:6px solid #3b82f6;}
.pq-ans{background:#f5f3ff;border-left:6px solid #8b5cf6;}
.pq-ans pre{background:#1f2937;color:#e5e7eb;padding:12px;border-radius:10px;}
.pq-map .pq-world{font-weight:800;margin:14px 0 6px;color:#7c3aed;font-size:16px;}
.pq-cell{padding:6px 10px;border-radius:8px;margin:3px 0;}
.pq-cell.done{background:#dcfce7;} .pq-cell.now{background:#fef9c3;font-weight:700;}
.pq-cell.open{background:#f1f5f9;} .pq-cell.lock{background:#f8fafc;color:#94a3b8;}
.pq-cert{background:linear-gradient(135deg,#f59e0b,#db2777);color:#fff;text-align:center;}
.pq-cert h1{font-size:34px;margin:6px 0;}
.pq-big{font-size:22px;font-weight:800;background:rgba(255,255,255,.2);
  display:inline-block;padding:6px 16px;border-radius:20px;}
.pq-cert hr{border-color:rgba(255,255,255,.4);}
</style>"""))


class _LoopRunaway(Exception):
    """Raised when a player's code runs too many steps (e.g. an endless loop)."""


import sys as _sys

class _loop_guard:
    """Context manager that stops runaway/infinite loops so Colab never freezes.
    It counts executed lines and bails out after a generous limit."""
    LIMIT = 500_000

    def __enter__(self):
        self.n = 0
        self._prev = _sys.gettrace()
        _sys.settrace(self._trace)
        return self

    def __exit__(self, *exc):
        _sys.settrace(self._prev)
        return False

    def _trace(self, frame, event, arg):
        self.n += 1
        if self.n > self.LIMIT:
            raise _LoopRunaway()
        return self._trace


def _mock_input(queue):
    """A safe stand-in for input() so kids' programs never freeze in a game.
    Returns preset answers one by one and echoes them like a real prompt."""
    q = list(queue) if queue else ["Alex"]
    box = {"i": 0}
    def _fake(prompt=""):
        val = q[box["i"]] if box["i"] < len(q) else q[-1]
        box["i"] += 1
        print(f"{prompt}{val}")
        return val
    return _fake


# ---------------------------------------------------------------
#  THE 27 LEVELS
# ---------------------------------------------------------------
# =====================================================================
#  PYTHONQUEST — LEVEL PACK  (27 levels, 7 worlds)
#  Each level teaches one idea and auto-checks the player's code.
#  A checker gets (ws, out):  ws = the code's variables,  out = printed text.
#  Starters are SCAFFOLDS with a TODO — running them as-is fails gently,
#  so kids actually write the key line themselves (that's how they learn).
# =====================================================================

LEVELS = [
# ==================== WORLD 1 : 🚀 BLAST OFF (print & strings) =========
dict(world="🚀 World 1 — Blast Off", emoji="🚀",
     title="Say Hello", xp=20, badge="🗣️ First Words",
     story="Every coder's first spell is <code>print()</code>. It makes the "
           "computer say things out loud on the screen!",
     concept="<p><code>print(\"...\")</code> shows whatever is inside the quotes.</p>"
             "<pre>print(\"Hi there!\")</pre>",
     task="Make the computer print exactly: <code>Hello, World!</code>",
     starter='print("")   # <- type  Hello, World!  between the quotes\n',
     hints=["Use the print function.",
            "Put your words inside \"double quotes\".",
            'Type it exactly: print("Hello, World!")'],
     answer='print("Hello, World!")',
     check=lambda ws, out: ok("You just cast your first spell! 🪄")
        if norm(out) == "hello, world!" else
        no("Print the words <b>Hello, World!</b> exactly (with the comma and !).")),

dict(world="🚀 World 1 — Blast Off", emoji="💬",
     title="Two Lines", xp=20,
     story="Each <code>print()</code> starts a brand new line. Let's introduce "
           "you to the computer!",
     concept="<p>Use <code>print()</code> more than once:</p>"
             "<pre>print(\"Line one\")\nprint(\"Line two\")</pre>",
     task="Print your name on the first line, then print your favourite animal "
          "on the second line (any words are fine — just two separate prints).",
     starter='# Line 1: your name. Line 2: your favourite animal.\n'
             'print("")\nprint("")\n',
     hints=["Write two print() lines, one under the other.",
            "The output should have exactly two lines of text."],
     answer='print("Sam")\nprint("Tiger")',
     check=lambda ws, out: ok("Two lines, two prints. Nice rhythm! 🎵")
        if len(lines(out)) >= 2 else
        no("I need <b>two</b> separate lines of text. Fill in both print() lines.")),

dict(world="🚀 World 1 — Blast Off", emoji="🔤",
     title="Glue Words Together", xp=25, badge="🔗 Word Wizard",
     story="Strings are just text. You can glue two strings with <code>+</code>, "
           "or drop values into a sentence with an <b>f-string</b>.",
     concept="<pre>name = \"Zoe\"\nprint(\"Hi \" + name)        # gluing\n"
             "print(f\"Hi {name}!\")        # f-string (easier!)</pre>",
     task="Make a variable <code>hero</code> with any name, then print "
          "<code>My hero is NAME</code> using an f-string.",
     starter='hero = ""      # <- put a name between the quotes\n'
             'print(f"My hero is {hero}")\n',
     hints=["First line: hero = \"SomeName\"",
            "Second line uses the f-string: print(f\"My hero is {hero}\")",
            "Note the little f right before the opening quote."],
     answer='hero = "Ada"\nprint(f"My hero is {hero}")',
     check=lambda ws, out: ok("f-strings will be your best friend! 💪")
        if "hero" in ws and norm(out).startswith("my hero is")
           and norm(str(ws["hero"])) in norm(out) and str(ws["hero"]).strip() != ""
        else no("Give <code>hero</code> a name and print "
                "<code>My hero is {hero}</code> with an f-string.")),

# ==================== WORLD 2 : 📦 MEMORY BOXES =======================
dict(world="📦 World 2 — Memory Boxes", emoji="📦",
     title="Make a Variable", xp=25,
     story="A variable is a labelled box that remembers a value for later.",
     concept="<pre>age = 10          # a box named age holding 10\n"
             "city = \"Cairo\"    # a box named city holding text</pre>",
     task="Create a variable named <code>score</code> equal to the number "
          "<code>100</code>.",
     starter="score = 0   # <- change 0 to 100\n",
     hints=["Use: score = 100", "No quotes — 100 is a number, not text."],
     answer="score = 100",
     check=lambda ws, out: ok("Box packed and labelled! 📦")
        if ws.get("score") == 100 else
        no("Set <code>score</code> to the number 100 (no quotes).")),

dict(world="📦 World 2 — Memory Boxes", emoji="🧮",
     title="Robot Calculator", xp=30, badge="🧮 Number Ninja",
     story="Python is a super calculator: <code>+ - * /</code>. Let's find the "
           "area of a room.",
     concept="<pre>width = 4\nheight = 5\narea = width * height   # 20</pre>",
     task="Make <code>width = 6</code> and <code>height = 7</code>, then make "
          "<code>area</code> equal to width times height. Print the area.",
     starter="width = 6\nheight = 7\narea = 0   # <- make this width * height\nprint(area)\n",
     hints=["Use * to multiply.", "area = width * height",
            "Then print(area) — it should be 42."],
     answer="width = 6\nheight = 7\narea = width * height\nprint(area)",
     check=lambda ws, out: ok("42 — the answer to everything! 🌌")
        if ws.get("area") == 42 else
        no("Make <code>area = width * height</code>. It should equal 42.")),

dict(world="📦 World 2 — Memory Boxes", emoji="🎤",
     title="Ask the Player", xp=30,
     story="<code>input()</code> asks the person a question and hands back what "
           "they type. (In this game it auto-answers <b>Alex</b> so nothing freezes.)",
     concept="<pre>name = input(\"Your name? \")\nprint(\"Hello \" + name)</pre>",
     task="Store <code>input(\"Name? \")</code> in a variable called "
          "<code>name</code>, then print <code>Welcome NAME!</code> with an f-string.",
     starter='name = ""   # <- use  input("Name? ")  instead of ""\n'
             'print(f"Welcome {name}!")\n',
     inputs=["Alex"],
     hints=["name = input(\"Name? \")",
            "print(f\"Welcome {name}!\")"],
     answer='name = input("Name? ")\nprint(f"Welcome {name}!")',
     check=lambda ws, out: ok("You just made an interactive program! 🎮")
        if ws.get("name") == "Alex" and "welcome alex" in norm(out) else
        no("Save <code>input(\"Name? \")</code> into <code>name</code>, then print "
           "<code>Welcome {name}!</code>.")),

dict(world="📦 World 2 — Memory Boxes", emoji="🔢",
     title="Words vs Numbers", xp=30,
     story="Typed answers arrive as <b>text</b>. To do maths, turn text into a "
           "number with <code>int()</code>.",
     concept="<pre>age_text = \"10\"\nage = int(age_text)   # now a number\n"
             "print(age + 5)        # 15</pre>",
     task="You have <code>age = int(\"12\")</code>. Make "
          "<code>next_year</code> equal to age + 1 and print it.",
     starter='age = int("12")\nnext_year = age   # <- add 1 to age\nprint(next_year)\n',
     hints=["int(\"12\") turns the text \"12\" into the number 12.",
            "next_year = age + 1 → should be 13."],
     answer='age = int("12")\nnext_year = age + 1\nprint(next_year)',
     check=lambda ws, out: ok("You tamed types! 🐉")
        if ws.get("next_year") == 13 else
        no("Make <code>next_year = age + 1</code> — it should be 13.")),

# ==================== WORLD 3 : 🔀 CROSSROADS (decisions) =============
dict(world="🔀 World 3 — Crossroads", emoji="⚖️",
     title="True or False?", xp=30, badge="⚖️ Logic Learner",
     story="Comparisons give a <b>boolean</b>: either <code>True</code> or "
           "<code>False</code>. <code>&gt; &lt; == != &gt;= &lt;=</code>",
     concept="<pre>print(10 > 3)     # True\nprint(5 == 6)     # False\n"
             "is_big = 100 > 50 # stores True</pre>",
     task="Make a variable <code>is_adult</code> that ends up <code>True</code> "
          "by writing a comparison such as <code>18 &gt;= 18</code>.",
     starter="is_adult = False   # <- replace with a comparison that is True, e.g. 18 >= 18\nprint(is_adult)\n",
     hints=["Use the >= comparison.", "is_adult = 18 >= 18 gives True."],
     answer="is_adult = 18 >= 18\nprint(is_adult)",
     check=lambda ws, out: ok("True dat! ✅")
        if ws.get("is_adult") is True else
        no("Make <code>is_adult</code> a comparison that results in True.")),

dict(world="🔀 World 3 — Crossroads", emoji="🚪",
     title="The Secret Door", xp=35,
     story="<code>if</code> runs code only when something is true. "
           "<code>else</code> runs when it isn't. Watch the indentation!",
     concept="<pre>password = \"open\"\nif password == \"open\":\n"
             "    print(\"Access granted\")\nelse:\n"
             "    print(\"Denied\")</pre>",
     task="Set <code>password = \"open\"</code>. If it equals <code>\"open\"</code>, "
          "print <code>Access granted</code>, otherwise print <code>Denied</code>.",
     starter='password = "open"\nif password == "open":\n    print("")   # <- print Access granted\nelse:\n    print("")   # <- print Denied\n',
     hints=["Line ends with a colon :  then indent the next line 4 spaces.",
            "Fill the first print with \"Access granted\"."],
     answer='password = "open"\nif password == "open":\n    print("Access granted")\nelse:\n    print("Denied")',
     check=lambda ws, out: ok("You unlocked the door! 🗝️")
        if "access granted" in norm(out) and "denied" not in norm(out) else
        no("It should print <b>Access granted</b> (and not Denied).")),

dict(world="🔀 World 3 — Crossroads", emoji="🎓",
     title="Grade Machine", xp=40, badge="🎯 Decision Maker",
     story="<code>elif</code> lets you check many cases in order — like a "
           "grading machine.",
     concept="<pre>if score >= 90:\n    grade = \"A\"\nelif score >= 70:\n"
             "    grade = \"B\"\nelse:\n    grade = \"C\"</pre>",
     task="Given <code>score = 85</code>, set <code>grade</code> to \"A\" if "
          "score&gt;=90, \"B\" if score&gt;=70, else \"C\". (It should become \"B\".)",
     starter='score = 85\nif score >= 90:\n    grade = "A"\nelif score >= 70:\n    grade = ""   # <- which grade goes here?\nelse:\n    grade = "C"\nprint(grade)\n',
     hints=["Order matters — check the biggest number first.",
            "85 is not >=90 but is >=70, so grade becomes \"B\"."],
     answer='score = 85\nif score >= 90:\n    grade = "A"\nelif score >= 70:\n    grade = "B"\nelse:\n    grade = "C"\nprint(grade)',
     check=lambda ws, out: ok("Grade-A logic! 🎓")
        if ws.get("grade") == "B" else
        no("With score 85 the grade should be <b>\"B\"</b>. Fill in the elif branch.")),

dict(world="🔀 World 3 — Crossroads", emoji="🎢",
     title="Ride Rules", xp=40,
     story="Combine conditions with <code>and</code> (both), <code>or</code> "
           "(either), <code>not</code> (flip).",
     concept="<pre>tall = True\nbrave = False\nprint(tall and brave)  # False\n"
             "print(tall or brave)   # True</pre>",
     task="Given <code>height = 140</code> and <code>has_ticket = True</code>, make "
          "<code>can_ride</code> True only if height &gt;= 120 <b>and</b> has_ticket.",
     starter="height = 140\nhas_ticket = True\ncan_ride = False   # <- use: height >= 120 and has_ticket\nprint(can_ride)\n",
     hints=["Use the word 'and' between the two conditions.",
            "can_ride = height >= 120 and has_ticket"],
     answer="height = 140\nhas_ticket = True\ncan_ride = height >= 120 and has_ticket\nprint(can_ride)",
     check=lambda ws, out: ok("Enjoy the ride! 🎢")
        if ws.get("can_ride") is True else
        no("Use <code>and</code> to combine both rules — result should be True.")),

# ==================== WORLD 4 : 🔁 LOOP LAND =========================
dict(world="🔁 World 4 — Loop Land", emoji="🔁",
     title="Count to Five", xp=35, badge="🔁 Loop Rookie",
     story="A <code>for</code> loop repeats code. <code>range(1, 6)</code> gives "
           "the numbers 1,2,3,4,5.",
     concept="<pre>for n in range(1, 6):\n    print(n)</pre>",
     task="Use a for loop with <code>range(1, 6)</code> to print the numbers "
          "1 to 5, each on its own line.",
     starter="for n in range(1, 1):   # <- fix the second number so you get 1..5\n    print(n)\n",
     hints=["range(1, 6) stops BEFORE 6, so it gives 1..5.",
            "Change the second number to 6."],
     answer="for n in range(1, 6):\n    print(n)",
     check=lambda ws, out: ok("Round and round! 🔁")
        if lines(out) == ["1", "2", "3", "4", "5"] else
        no("Print 1,2,3,4,5 each on its own line using range(1, 6).")),

dict(world="🔁 World 4 — Loop Land", emoji="👫",
     title="Greet the Squad", xp=40,
     story="You can loop over a <b>list</b> of items directly — no numbers needed.",
     concept="<pre>friends = [\"Mia\", \"Leo\"]\nfor f in friends:\n"
             "    print(\"Hi \" + f)</pre>",
     task="Loop over <code>friends = [\"Mia\", \"Leo\", \"Sam\"]</code> and print "
          "<code>Hi NAME</code> for each one.",
     starter='friends = ["Mia", "Leo", "Sam"]\nfor f in friends:\n    print("")   # <- print  Hi  and the name f\n',
     hints=["Inside the loop: print(\"Hi \" + f)",
            "Or use an f-string: print(f\"Hi {f}\")"],
     answer='friends = ["Mia", "Leo", "Sam"]\nfor f in friends:\n    print("Hi " + f)',
     check=lambda ws, out: ok("The whole squad says hi! 👋")
        if all(f"hi {n}" in norm(out) for n in ["mia", "leo", "sam"]) else
        no("Loop the list and print <b>Hi Mia</b>, <b>Hi Leo</b>, <b>Hi Sam</b>.")),

dict(world="🔁 World 4 — Loop Land", emoji="➕",
     title="Add Them Up", xp=45, badge="🧠 Loop Thinker",
     story="A loop can build up an answer step by step. Start a total at 0 and "
           "add to it each time.",
     concept="<pre>total = 0\nfor n in [10, 20, 30]:\n    total = total + n\n"
             "print(total)   # 60</pre>",
     task="Start <code>total = 0</code>, then loop over "
          "<code>[5, 10, 15, 20]</code> adding each number to total. Print total "
          "(it should be 50).",
     starter="total = 0\nfor n in [5, 10, 15, 20]:\n    total = total   # <- add n to total here\nprint(total)\n",
     hints=["Make total = 0 BEFORE the loop.",
            "Inside: total = total + n (or total += n)."],
     answer="total = 0\nfor n in [5, 10, 15, 20]:\n    total += n\nprint(total)",
     check=lambda ws, out: ok("You're an adding machine! 🧮")
        if ws.get("total") == 50 else
        no("Sum the list into <code>total</code> — it should be 50.")),

dict(world="🔁 World 4 — Loop Land", emoji="🚀",
     title="Countdown!", xp=45,
     story="A <code>while</code> loop repeats <b>as long as</b> something is "
           "true. Perfect for a rocket countdown.",
     concept="<pre>n = 3\nwhile n > 0:\n    print(n)\n    n = n - 1\n"
             "print(\"Go!\")</pre>",
     task="Start <code>n = 5</code>. While n &gt; 0, print n then subtract 1. "
          "After the loop, print <code>Blast off!</code>.",
     starter='n = 5\nwhile n > 0:\n    print(n)\n    # <- add a line here that lowers n by 1, or the loop never stops!\nprint("Blast off!")\n',
     hints=["Add  n = n - 1  inside the loop, or it runs forever!",
            "Print \"Blast off!\" AFTER the loop (not indented)."],
     answer='n = 5\nwhile n > 0:\n    print(n)\n    n = n - 1\nprint("Blast off!")',
     check=lambda ws, out: ok("🚀 Liftoff!")
        if lines(out)[:5] == ["5","4","3","2","1"] and "blast off" in norm(out) else
        no("Count 5,4,3,2,1 then print <b>Blast off!</b>. Remember to lower n each loop.")),

# ==================== WORLD 5 : 🗃️ TREASURE CHESTS ===================
dict(world="🗃️ World 5 — Treasure Chests", emoji="🗃️",
     title="Pack a List", xp=40, badge="📋 List Keeper",
     story="A <b>list</b> holds many things in order. Counting starts at <b>0</b>!",
     concept="<pre>fruits = [\"apple\", \"kiwi\", \"mango\"]\n"
             "print(fruits[0])   # apple\nprint(fruits[2])   # mango</pre>",
     task="Make a list <code>colors</code> with \"red\", \"green\", \"blue\". "
          "Print the <b>first</b> colour using <code>colors[0]</code>.",
     starter='colors = ["red", "green", "blue"]\nprint(colors[1])   # <- which index is the FIRST item?\n',
     hints=["The first item is colors[0], not colors[1].",
            "Lists start counting at 0."],
     answer='colors = ["red", "green", "blue"]\nprint(colors[0])',
     check=lambda ws, out: ok("Zero is the start — you got it! 0️⃣")
        if isinstance(ws.get("colors"), list) and len(ws["colors"]) == 3
           and "red" in norm(out) and "green" not in norm(out) else
        no("Print <code>colors[0]</code> — the FIRST colour, which is red.")),

dict(world="🗃️ World 5 — Treasure Chests", emoji="➕",
     title="Grow the List", xp=45,
     story="Lists can grow with <code>.append()</code>, and "
           "<code>len()</code> counts how many items are inside.",
     concept="<pre>bag = [\"gold\"]\nbag.append(\"gem\")\nprint(len(bag))  # 2</pre>",
     task="Start <code>bag = [\"gold\"]</code>, append <code>\"gem\"</code> and "
          "<code>\"key\"</code>, then print <code>len(bag)</code> (should be 3).",
     starter='bag = ["gold"]\n# <- add two lines: bag.append("gem")  and  bag.append("key")\nprint(len(bag))\n',
     hints=["Call bag.append(\"gem\") then bag.append(\"key\") on their own lines.",
            "len(bag) counts the items — you want 3."],
     answer='bag = ["gold"]\nbag.append("gem")\nbag.append("key")\nprint(len(bag))',
     check=lambda ws, out: ok("Treasure collected! 💎")
        if isinstance(ws.get("bag"), list) and len(ws["bag"]) == 3 else
        no("Append two items so the bag has 3, then print len(bag).")),

dict(world="🗃️ World 5 — Treasure Chests", emoji="🔢",
     title="Square Factory", xp=50, badge="🏭 Data Builder",
     story="Loop + list = a factory. Build a new list of squared numbers.",
     concept="<pre>squares = []\nfor n in [1, 2, 3]:\n    squares.append(n * n)\n"
             "print(squares)   # [1, 4, 9]</pre>",
     task="Build a list <code>squares</code> of the squares of 1,2,3,4,5. "
          "The result should be <code>[1, 4, 9, 16, 25]</code>.",
     starter="squares = []\nfor n in range(1, 6):\n    squares.append(n)   # <- append the SQUARE of n (n * n)\nprint(squares)\n",
     hints=["Start with an empty list: squares = []",
            "Inside the loop: squares.append(n * n)"],
     answer="squares = []\nfor n in range(1, 6):\n    squares.append(n * n)\nprint(squares)",
     check=lambda ws, out: ok("Factory running at full power! 🏭")
        if ws.get("squares") == [1, 4, 9, 16, 25] else
        no("Build [1, 4, 9, 16, 25] by appending n*n in a loop.")),

dict(world="🗃️ World 5 — Treasure Chests", emoji="🗂️",
     title="The Name Tag", xp=50,
     story="A <b>dictionary</b> stores pairs: a <b>key</b> and its <b>value</b>. "
           "Great for describing one thing.",
     concept="<pre>hero = {\"name\": \"Zed\", \"hp\": 100}\n"
             "print(hero[\"name\"])   # Zed</pre>",
     task="Make a dictionary <code>pet</code> with keys <code>\"name\"</code> "
          "(any name) and <code>\"legs\"</code> = 4. Print the pet's name using "
          "<code>pet[\"name\"]</code>.",
     starter='pet = {"name": "", "legs": 0}   # <- add a name, and set legs to 4\nprint(pet["name"])\n',
     hints=["Fill in a name and change legs to 4.",
            "Access a value with pet[\"name\"]."],
     answer='pet = {"name": "Rex", "legs": 4}\nprint(pet["name"])',
     check=lambda ws, out: ok("Key + value = power! 🗝️")
        if isinstance(ws.get("pet"), dict) and ws["pet"].get("legs") == 4
           and str(ws["pet"].get("name", "")).strip() != ""
           and norm(str(ws["pet"]["name"])) in norm(out) else
        no("Give <code>pet</code> a name and legs=4, then print its name.")),

dict(world="🗃️ World 5 — Treasure Chests", emoji="📖",
     title="Read the Scores", xp=55, badge="🗂️ Dict Master",
     story="Loop over a dictionary with <code>.items()</code> to see every key "
           "and value.",
     concept="<pre>scores = {\"Mia\": 8, \"Leo\": 5}\n"
             "for name, pts in scores.items():\n    print(name, pts)</pre>",
     task="Given <code>scores = {\"Mia\": 8, \"Leo\": 5}</code>, loop with "
          "<code>.items()</code> and print <code>NAME scored PTS</code> for each.",
     starter='scores = {"Mia": 8, "Leo": 5}\nfor name, pts in scores.items():\n    print("")   # <- print   NAME scored PTS   using an f-string\n',
     hints=["for name, pts in scores.items():",
            "print(f\"{name} scored {pts}\")"],
     answer='scores = {"Mia": 8, "Leo": 5}\nfor name, pts in scores.items():\n    print(f"{name} scored {pts}")',
     check=lambda ws, out: ok("You read the whole scoreboard! 📊")
        if "mia scored 8" in norm(out) and "leo scored 5" in norm(out) else
        no("Loop with .items() and print <b>Mia scored 8</b> and <b>Leo scored 5</b>.")),

# ==================== WORLD 6 : ⚙️ MACHINE SHOP (functions) ==========
dict(world="⚙️ World 6 — Machine Shop", emoji="⚙️",
     title="Build a Machine", xp=45, badge="⚙️ Machine Maker",
     story="A <b>function</b> is a reusable machine you build once and use "
           "again and again with <code>def</code>.",
     concept="<pre>def greet(name):\n    print(\"Hello \" + name)\n\n"
             "greet(\"Mia\")   # Hello Mia</pre>",
     task="Define a function <code>greet(name)</code> that prints "
          "<code>Hello NAME</code>. Then call <code>greet(\"World\")</code>.",
     starter='def greet(name):\n    print("")   # <- print  Hello  and the name\n\ngreet("World")\n',
     hints=["Inside the function: print(\"Hello \" + name)",
            "The call greet(\"World\") should print Hello World."],
     answer='def greet(name):\n    print("Hello " + name)\n\ngreet("World")',
     check=lambda ws, out: ok("Your first machine works! ⚙️")
        if callable(ws.get("greet")) and "hello world" in norm(out) else
        no("Define <code>greet(name)</code> that prints Hello NAME, then call it.")),

dict(world="⚙️ World 6 — Machine Shop", emoji="↩️",
     title="Send Back an Answer", xp=50,
     story="<code>return</code> hands a value <b>back</b> so you can store or "
           "reuse it — much more powerful than just printing.",
     concept="<pre>def add(a, b):\n    return a + b\n\n"
             "total = add(2, 3)   # total is 5</pre>",
     task="Write a function <code>add(a, b)</code> that <b>returns</b> a + b. "
          "(The game will test it with different numbers.)",
     starter="def add(a, b):\n    return 0   # <- return a + b instead of 0\n",
     hints=["Use return, not print.", "return a + b"],
     answer="def add(a, b):\n    return a + b",
     check=lambda ws, out: ok("Machines that answer back — pro level! 🏆")
        if callable(ws.get("add")) and ws["add"](2, 3) == 5
           and ws["add"](10, 20) == 30 else
        no("Make <code>add(a, b)</code> <b>return</b> a + b (use return, not print).")),

dict(world="⚙️ World 6 — Machine Shop", emoji="⚖️",
     title="The Even Detector", xp=55, badge="🔬 Function Scientist",
     story="Functions can make decisions and return True/False. "
           "<code>%</code> gives the remainder — even numbers leave remainder 0.",
     concept="<pre>def is_even(n):\n    return n % 2 == 0\n\n"
             "print(is_even(4))   # True</pre>",
     task="Write <code>is_even(n)</code> that returns <code>True</code> when n is "
          "even, else <code>False</code>. (Use <code>n % 2 == 0</code>.)",
     starter="def is_even(n):\n    return False   # <- return  n % 2 == 0\n",
     hints=["n % 2 is the remainder when dividing by 2.",
            "return n % 2 == 0"],
     answer="def is_even(n):\n    return n % 2 == 0",
     check=lambda ws, out: ok("Detector calibrated! 🔬")
        if callable(ws.get("is_even")) and ws["is_even"](4) is True
           and ws["is_even"](7) is False else
        no("Return <code>n % 2 == 0</code> so even→True, odd→False.")),

dict(world="⚙️ World 6 — Machine Shop", emoji="🔤",
     title="Vowel Counter", xp=60, badge="🧙 Code Combiner",
     story="Time to combine EVERYTHING: a function, a loop, an if, and a "
           "counter. Count the vowels in a word.",
     concept="<pre>def count_vowels(word):\n    count = 0\n"
             "    for letter in word:\n        if letter in \"aeiou\":\n"
             "            count = count + 1\n    return count</pre>",
     task="Write <code>count_vowels(word)</code> returning how many vowels "
          "(a, e, i, o, u) are in the lowercase word. E.g. "
          "<code>count_vowels(\"banana\")</code> → 3.",
     starter='def count_vowels(word):\n    count = 0\n    for letter in word:\n        # <- if the letter is a vowel, add 1 to count\n        pass\n    return count\n',
     hints=["Loop each letter; use  if letter in \"aeiou\":  then count += 1.",
            "Delete the 'pass' line once you add your if.",
            "Return count at the end (not indented inside the loop)."],
     answer='def count_vowels(word):\n    count = 0\n    for letter in word:\n        if letter in "aeiou":\n            count += 1\n    return count',
     check=lambda ws, out: ok("You combined 4 ideas into 1 machine. Wizard! 🧙")
        if callable(ws.get("count_vowels")) and ws["count_vowels"]("banana") == 3
           and ws["count_vowels"]("sky") == 0
           and ws["count_vowels"]("aeiou") == 5 else
        no("Count letters that are in \"aeiou\". \"banana\"→3, \"sky\"→0.")),

# ==================== WORLD 7 : 🏆 BOSS BUILD (real project) =========
dict(world="🏆 World 7 — Boss Build: Quiz Bot", emoji="🧩",
     title="Design the Quiz", xp=60, badge="🏗️ Project Architect",
     story="For your final quest you'll build a real <b>Quiz Bot</b> game — "
           "piece by piece. First, design the questions as data.",
     concept="A list of dictionaries is perfect for a quiz:"
             "<pre>quiz = [\n  {\"q\": \"2+2?\", \"a\": \"4\"},\n"
             "  {\"q\": \"Sky colour?\", \"a\": \"blue\"},\n]</pre>",
     task="Create a list called <code>quiz</code> with <b>at least 2</b> "
          "dictionaries, each having a <code>\"q\"</code> (question) and "
          "<code>\"a\"</code> (answer) key.",
     starter='quiz = [\n    {"q": "What is 2 + 2?", "a": "4"},\n    # <- add one more question dict here, with its own "q" and "a"\n]\nprint(len(quiz), "questions ready!")\n',
     hints=["Copy the first line and change the question and answer.",
            "Each dict needs a \"q\" and an \"a\" key. You need 2 or more."],
     answer='quiz = [\n    {"q": "What is 2 + 2?", "a": "4"},\n    {"q": "What colour is the sky?", "a": "blue"},\n]\nprint(len(quiz), "questions ready!")',
     check=lambda ws, out: ok("Your quiz data is ready to power the game! 🏗️")
        if isinstance(ws.get("quiz"), list) and len(ws["quiz"]) >= 2
           and all(isinstance(q, dict) and "q" in q and "a" in q for q in ws["quiz"])
        else no("Make a <code>quiz</code> list of 2+ dicts, each with \"q\" and \"a\".")),

dict(world="🏆 World 7 — Boss Build: Quiz Bot", emoji="✅",
     title="The Answer Checker", xp=65,
     story="Now build the brain: a function that checks one answer and returns "
           "<code>True</code> or <code>False</code>. Ignore capital letters and "
           "spaces so players aren't punished for typing style.",
     concept="<pre>def check(guess, correct):\n"
             "    return guess.strip().lower() == correct.strip().lower()</pre>",
     task="Write <code>check(guess, correct)</code> that returns True when the "
          "two answers match after <code>.strip().lower()</code>. "
          "<code>check(\" Blue \", \"blue\")</code> must be True.",
     starter="def check(guess, correct):\n    return False   # <- compare guess and correct after .strip().lower()\n",
     hints=[".strip() removes spaces, .lower() makes lowercase.",
            "return guess.strip().lower() == correct.strip().lower()"],
     answer="def check(guess, correct):\n    return guess.strip().lower() == correct.strip().lower()",
     check=lambda ws, out: ok("Fair and smart — great judging! ⚖️")
        if callable(ws.get("check")) and ws["check"](" Blue ", "blue") is True
           and ws["check"]("cat", "dog") is False else
        no("Return True when guess matches correct after .strip().lower().")),

dict(world="🏆 World 7 — Boss Build: Quiz Bot", emoji="🏆",
     title="Run the Whole Game!", xp=120, badge="🏆 Quiz Bot Champion",
     story="THE FINAL BOSS! Put it all together: loop through the quiz, ask "
           "each question, keep <b>score</b>, and give a final result. You've "
           "learned every piece — now assemble the machine! 🚀",
     concept="Use your <code>quiz</code> data and everything you know: a loop, "
             "<code>input()</code>, an <code>if</code>, a counter, and an "
             "f-string.<pre>score = 0\nfor item in quiz:\n"
             "    guess = input(item[\"q\"] + \" \")\n"
             "    if guess.strip().lower() == item[\"a\"].strip().lower():\n"
             "        score += 1\nprint(f\"You scored {score}/{len(quiz)}\")</pre>"
             "<i>(The game auto-answers each question so it runs instantly.)</i>",
     task="Using the same <code>quiz</code> from before, loop through it, ask "
          "each <code>item[\"q\"]</code> with input(), add 1 to <code>score</code> "
          "for each correct answer, and finally print "
          "<code>You scored SCORE/TOTAL</code>. Store the number in "
          "<code>score</code>.",
     starter='quiz = [\n    {"q": "What is 2 + 2?", "a": "4"},\n    {"q": "What colour is the sky?", "a": "blue"},\n]\n\nscore = 0\nfor item in quiz:\n    guess = input(item["q"] + " ")\n    # <- if the cleaned guess matches item["a"], add 1 to score\n\nprint(f"You scored {score}/{len(quiz)}")\n',
     inputs=["4", "blue", "4", "blue", "4", "blue"],
     hints=["Inside the loop compare guess.strip().lower() with item[\"a\"].strip().lower().",
            "When they match:  score += 1",
            "The final print is already written for you."],
     answer='quiz = [\n    {"q": "What is 2 + 2?", "a": "4"},\n    {"q": "What colour is the sky?", "a": "blue"},\n]\nscore = 0\nfor item in quiz:\n    guess = input(item["q"] + " ")\n    if guess.strip().lower() == item["a"].strip().lower():\n        score += 1\nprint(f"You scored {score}/{len(quiz)}")',
     check=lambda ws, out: ok("🏆 YOU BUILT A REAL GAME! Legendary!")
        if isinstance(ws.get("quiz"), list) and ws.get("score") == len(ws["quiz"])
           and "you scored" in norm(out) and f"/{len(ws['quiz'])}" in out else
        no("Loop the quiz, count correct answers into <code>score</code>, and print "
           "<code>You scored score/total</code>. (The auto-player answers correctly, "
           "so score should equal the number of questions.)")),
]


print("✅ Game engine loaded with", len(LEVELS),
      "levels! Now run STEP 2 below to play. 🚀")


In [ ]:
# @title 🕹️ STEP 2 — Play!  { display-mode: "form" }
# 1) Change the name below to YOUR name.
# 2) Press the ▶ play button. Your adventure begins!

my_name = "Coder"   # <-- put your name here, keep the quotes

quest = PythonQuest(LEVELS, player=my_name)
quest.start()


### 🧭 Extra commands

Run this cell any time to see your map, or to reset your progress.


In [ ]:
# 🧭 HANDY COMMANDS — run any of these in a new cell whenever you like:

quest.map()      # 🗺️  see the whole adventure map & your progress
# quest.start()  # ▶️  jump back into the current level
# quest.reset()  # 🧹  erase progress and start the whole quest over


---

# 🚀 BUILD ZONE — now make your OWN things!

**Congratulations, coder!** 🎉 You finished the quest. In the game you typed in
the little black box. To *build real things*, you write Python in **normal Colab
cells** — exactly the same Python you just learned.

**How to use a real code cell:**
- Click **`+ Code`** at the top to make a new cell (or use the ones below).
- Type your code, then press **▶** (or **Shift + Enter**) to run it.
- Here, `input()` is real — the program will actually wait for you to type! ⌨️

Below are **5 starter projects**. For each one:
1. ▶️ **Run it** and play.
2. ✏️ Find the lines marked `# ✏️` and **change them** to make it yours.
3. 💥 **Break it, fix it, remix it** — that's how real coders learn!

Every project uses only things you learned in the game: `print`, variables,
`input`, `if/elif/else`, loops, lists, dictionaries and functions. 💪


In [ ]:
# 🎲 PROJECT 1 — Guess My Number
# Skills you already know: variables, while loop, if/elif/else, input.
import random

secret = random.randint(1, 20)   # ✏️ change 20 to make it harder or easier
guesses = 0
print("🎲 I'm thinking of a number from 1 to 20. Can you guess it?")

while True:
    guess = int(input("Your guess: "))
    guesses = guesses + 1
    if guess < secret:
        print("⬆️ Too low! Try higher.")
    elif guess > secret:
        print("⬇️ Too high! Try lower.")
    else:
        print(f"🎉 YES! You got it in {guesses} guesses!")
        break


In [ ]:
# 📖 PROJECT 2 — Silly Story Maker (Mad Libs)
# Skills: input, variables, f-strings.
name   = input("A name: ")
animal = input("An animal: ")
place  = input("A place: ")
food   = input("A food: ")

# ✏️ Change the story to anything you want!
print()
print("📖 Here is your silly story:")
print(f"One day, {name} rode a giant {animal} all the way to {place}.")
print(f"They were so hungry they ate {food} until the sun went down. The End! 🌅")


In [ ]:
# ✊✋✌️ PROJECT 3 — Rock, Paper, Scissors
# Skills: functions, lists, dictionaries, random, if/elif/else, loops.
import random

def play_round():
    options = ["rock", "paper", "scissors"]
    you = input("Choose rock, paper or scissors: ").strip().lower()
    computer = random.choice(options)
    print(f"🤖 Computer chose {computer}")
    if you == computer:
        return "tie"
    beats = {"rock": "scissors", "paper": "rock", "scissors": "paper"}
    if beats.get(you) == computer:
        return "you"
    return "computer"

score = {"you": 0, "computer": 0}
for round_number in range(3):        # ✏️ play more rounds by changing 3
    print(f"\n--- Round {round_number + 1} ---")
    result = play_round()
    if result == "you":
        print("✅ You win this round!"); score["you"] += 1
    elif result == "computer":
        print("❌ Computer wins this round."); score["computer"] += 1
    else:
        print("🤝 It's a tie!")

print(f"\n🏁 Final — You: {score['you']}, Computer: {score['computer']}")


In [ ]:
# 🧮 PROJECT 4 — Pocket-Money Saver
# Skills: input, int, maths, if/else, f-strings.
weekly = int(input("How much pocket money do you get each week? "))
weeks  = int(input("How many weeks will you save? "))
goal   = int(input("How much does the thing you want cost? "))

saved = weekly * weeks
print(f"\n💰 In {weeks} weeks you'll have saved {saved}.")
if saved >= goal:
    print(f"🎉 That's enough for your {goal} goal — go for it!")
else:
    print(f"😮 You'll still need {goal - saved} more. Try saving longer!")


In [ ]:
# 🤖 PROJECT 5 — Your OWN Quiz Bot (the big one!)
# Skills: EVERYTHING — lists, dictionaries, functions, loops, input, if, f-strings.

# ✏️ Add, remove or change these questions — make the quiz about YOUR topic!
quiz = [
    {"q": "What is the capital of France? ", "a": "paris"},
    {"q": "How many legs does a spider have? ", "a": "8"},
    {"q": "What planet do we live on? ", "a": "earth"},
]

def ask(question, correct):
    guess = input(question).strip().lower()
    return guess == correct.strip().lower()

score = 0
for item in quiz:
    if ask(item["q"], item["a"]):
        print("✅ Correct!")
        score += 1
    else:
        print(f"❌ Nope — the answer was {item['a']}.")

print(f"\n🏆 You scored {score} out of {len(quiz)}!")
if score == len(quiz):
    print("Perfect score — you're a genius! 🌟")


---

## 🧠 Your Turn — challenges to level up

Pick one and build it using the projects above as a starting point:

- 🎨 **Make Project 2 your own story** with 6+ blanks and a twist ending.
- 🏆 **Turn Project 5 into a quiz about your favourite game, animal or team** —
  add 8 questions and a "You are a...!" result at the end based on the score.
- 🔢 **Add difficulty to Project 1**: let the player pick Easy (1–10) or Hard (1–50).
- ✂️ **Best-of-5 Rock Paper Scissors**: keep playing until someone wins 3 rounds.
- 🤖 **Build a tiny chatbot**: ask the user questions in a loop and reply with
  `if/elif` depending on what they type. Type "bye" to stop.

**Where to go next**
- Try Python's `turtle` drawing, or make a to-do list with a `list`.
- Free practice: <https://www.codewars.com>, <https://www.checkio.org>,
  or search "Python for kids projects".

> 🌟 The secret to becoming a real coder: **change working code and see what
> happens.** Every bug you fix makes you stronger.


---

## 👩‍🏫 For the teacher / parent — a 1-week plan

Kids learn each idea, then **immediately use it** in a puzzle that auto-checks
their code and celebrates every win. No memorising definitions — they *do* it.

| Day | Worlds | They learn to… |
|----|---------|----------------|
| **1** | 🚀 World 1 | print, strings, f-strings — make the computer talk |
| **2** | 📦 World 2 | variables, numbers/maths, `input()`, types |
| **3** | 🔀 World 3 | booleans, `if/elif/else`, `and/or/not` — decisions |
| **4** | 🔁 World 4 | `for`, `range`, `while` loops, building totals |
| **5** | 🗃️ World 5 | lists & dictionaries — storing lots of data |
| **6** | ⚙️ World 6 | **functions** — building reusable machines |
| **7** | 🏆 World 7 | **build a real Quiz Bot game** from scratch! |
| **7+** | 🚀 Build Zone | 5 real projects to run, edit and remix in normal cells |

**By the end** kids can read code, use variables, make decisions, loop, store
data, write their own functions, and assemble a working program — enough to
build something real of their own. 🎓

**The bridge to real building:** after the game, the **🚀 Build Zone** cells show
kids that the *same* Python works in ordinary Colab cells (where `input()` runs
for real). They run a project, change the `# ✏️` lines, and remix it — this is
where "I learned Python" becomes "I can build things." Great for Day 7 and the
whole second week.

**Tips for running it**
- One notebook per child (File → *Save a copy in Drive*). Progress saves per session.
- ~4 levels a day keeps it fun and un-rushed; let fast learners race ahead.
- The **💡 Hint** ladder is designed so kids solve it themselves — nudge them to
  hints before answers.
- After World 7, challenge them: *"Now change the quiz questions to your own!"* —
  editing working code is where confidence really grows.

*Built with ❤️ to make first steps in Python feel like play.*
